# Modul 1 - Fondasi Pelatihan Artificial Neural Network (Multi Layer Perceptron)

## Pokok Bahasan
1. Persiapan Data Tabular (Dataset Diabetes)
2. Membangun Arsitektur MLP dan Inisialisasi Bobot
3. Loss Function, Optimizer, dan Siklus Pelatihan
4. Evaluasi dan Metrik Kinerja

Modul ini berfokus pada pemahaman fundamental alur pelatihan Multi-Layer Perceptron (MLP) pada data tabular. Setiap mahasiswa akan membedah proses penyiapan arsitektur model, inisialisasi bobot awal, hingga pemilihan Loss Function dan Optimizer yang tepat untuk klasifikasi biner.

Modul ini akan menggunakan dataset medis klasik untuk memprediksi probabilitas pasien mengidap diabetes berdasarkan indikator kesehatannya.

## 1. Persiapan Data Tabular (Dataset Diabetes)

In [ ]:
# Library yang digunakan untuk manipulasi dan analisis data
import pandas as pd
import numpy as np
# Library untuk penerapan deep learning berbasis pytorch
import torch
import torch.nn as nn
import torch.optim as optim
# Library yang digunakan untuk pra-pemrosesan data & evaluasi model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
# Library untuk visualisasi data
import matplotlib.pyplot as plt

In [ ]:
# Load Data (Memuat dataset Pima Indians Diabetes dari repository publik)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
kolom = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
         'Insulin', 'BMI', 'DiabetesPedigree', 'Age', 'Outcome']
df = pd.read_csv(url, names=kolom)
df

In [ ]:
# Pemisahan Fitur (X) dan Target (y)
X = df.drop('Outcome', axis=1).values
print(X)
y = df['Outcome'].values
print(y)

In [ ]:
# Train/Test Split (80% Latih, 20% Uji)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                    random_state=42)
print(f"Jumlah data latih: {len(X_train)}")
print(f"Jumlah data uji: {len(X_test)}")

In [ ]:
# Feature Scaling (Z-Score Normalization)
# Jaringan Saraf Tiruan sangat sensitif terhadap skala fitur, maka harus
# disamakan skalanya. Selain itu perlu diingat bahwa pada test_set hanya
# melakukan transform (pengubahan), karena sudah dilakukan fit (penyesuaian)
# terhadap train_set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Sample data latih setelah scaling:\n{X_train_scaled[:5]}")

In [ ]:
# Konversi ke PyTorch Tensor, dengan pertimbangan matriks (baris, kolom)
# Maka itu reshape y perlu dilakukan agar tercipta (n baris data, 1 kolom label)
X_train_t = torch.FloatTensor(X_train_scaled)
y_train_t = torch.FloatTensor(y_train).view(-1, 1)
X_test_t = torch.FloatTensor(X_test_scaled)
y_test_t = torch.FloatTensor(y_test).view(-1, 1)
print(f"Data latih siap!")
print(f"Ukuran X_train: {X_train_t.shape}")
print(f"Ukuran y_train: {y_train_t.shape}")

## 2. Membangun Arsitektur MLP dan Inisialisasi Bobot

Dalam mendefinisikan jaringan, akan digunakan `nn.Module`. Pemilihan Activation Function (seperti ReLU) di setiap lapisan tersembunyi (hidden layer) berfungsi untuk menangkap pola non-linear.

**Weight Initialization (Inisialisasi Bobot):** Bobot tidak boleh dimulai dari angka nol. Pada modul akan digunakan Xavier/Glorot Initialization yang dirancang khusus agar distribusi varians tetap stabil dari lapisan ke lapisan, sehingga model belajar lebih cepat.

In [ ]:
# Mendefinisikan Arsitektur MLP
class DiabetesMLP(nn.Module):
  def __init__(self, input_dim):
    super(DiabetesMLP, self).__init__()
    # Lapisan 1: Input ke Hidden Layer 1 (16 neuron)
    self.layer1 = nn.Linear(input_dim, 16)
    # Lapisan 2: Hidden 1 ke Hidden 2 (8 neuron)
    self.layer2 = nn.Linear(16, 8)
    # Lapisan 3: Hidden 2 ke Output (1 neuron)
    self.layer3 = nn.Linear(8, 1)
    # Aktivasi ReLU (Rectified Linear Unit) -> Mengubah nilai negatif menjadi 0
    self.relu = nn.ReLU()
    # Menerapkan Inisialisasi Bobot (Xavier Uniform)
    nn.init.xavier_uniform_(self.layer1.weight)
    nn.init.xavier_uniform_(self.layer2.weight)
    nn.init.xavier_uniform_(self.layer3.weight)

  def forward(self, x):
    # Alur data (Forward Pass)
    x = self.relu(self.layer1(x))
    x = self.relu(self.layer2(x))
    # Output berupa nilai mentah (logits), belum dilewatkan sigmoid
    x = self.layer3(x)
    return x

In [ ]:
# Inisialisasi Model
input_dim = X_train_t.shape[1] # 8 fitur
model = DiabetesMLP(input_dim)
print(model)

## 3. Loss Function, Optimizer, dan Siklus Pelatihan

**Loss Function:** Karena ini masalah klasifikasi biner, maka akan digunakan Binary Cross-Entropy. Di PyTorch, `nn.BCEWithLogitsLoss()` sangat disarankan karena menggabungkan fungsi Sigmoid dan BCE secara otomatis dalam satu komputasi yang secara numerik jauh lebih stabil.

**Optimizer:** `optim.Adam` adalah algoritma yang cerdas. Optimizer ini menyesuaikan learning rate secara dinamis untuk setiap parameter berdasarkan momentum (moving average dari gradien sebelumnya).

In [ ]:
# Definisi Loss Function dan Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# Proses pelatihan model
epochs = 150
train_losses, val_losses = [], []

print("--- Memulai Pelatihan ---")
for epoch in range(epochs):
  # Mode Pelatihan
  model.train()
  # 1. Bersihkan sisa gradien dari iterasi sebelumnya
  optimizer.zero_grad()
  # 2. Forward Pass: Hitung prediksi
  pred_train = model(X_train_t)
  # 3. Hitung Kesalahan (Loss)
  loss_train = criterion(pred_train, y_train_t)
  # 4. Backpropagation: Hitung turunan
  loss_train.backward()
  # 5. Update Bobot
  optimizer.step()

  # Mode Evaluasi (Validasi)
  model.eval()
  with torch.no_grad(): # Matikan perhitungan gradien agar hemat memori
    pred_val = model(X_test_t)
    loss_val = criterion(pred_val, y_test_t)

  # Menyimpan sejarah loss
  train_losses.append(loss_train.item())
  val_losses.append(loss_val.item())

  if (epoch+1) % 30 == 0:
    print(f"Epoch {epoch+1:3d}/{epochs}")
    print(f"Train Loss: {loss_train.item():.4f} | Val Loss: {loss_val.item():.4f}\n")

print("--- Pelatihan Selesai ---")

## 4. Evaluasi dan Metrik Kinerja

Setelah pelatihan, proses dilanjutkan dengan mengevaluasi model pada set pengujian menggunakan metrik nyata seperti Accuracy dan Recall (sebagai contoh adalah kemampuan model untuk mendeteksi orang yang benar-benar sakit).

In [ ]:
# Visualisasi kurva pelatihan
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Kurva Loss: Pelatihan vs Validasi')
plt.xlabel('Epochs')
plt.ylabel('BCE Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Menghitung performance metrics
model.eval()
with torch.no_grad():
  raw_output = model(X_test_t)
  # Terapkan Sigmoid untuk mendapat probabilitas, lalu bulatkan ke 0 atau 1 (Threshold 0.5)
  y_pred_probs = torch.sigmoid(raw_output)
  y_pred_classes = (y_pred_probs >= 0.5).float()

y_true_np = y_test_t.numpy()
y_pred_np = y_pred_classes.numpy()

acc = accuracy_score(y_true_np, y_pred_np)
prec = precision_score(y_true_np, y_pred_np)
rec = recall_score(y_true_np, y_pred_np)

print("--- METRIK KINERJA PADA DATA UJI ---")
print(f"Akurasi   : {acc * 100:.2f}% (Persentase tebakan benar secara keseluruhan)")
print(f"Precision : {prec * 100:.2f}% (Bila model menebak, berapa persen tebakannya benar?)")
print(f"Recall    : {rec * 100:.2f}% (Dari semua data, berapa persen berhasil dideteksi?)")